_____________________________________________________________________________________________________________________
## __Краткое инфо__

### __Криптовалюта:__ ETH

### __Сайт:__ Investing.com

### __Период:__ 01.11.2024 - 01.11.2025

### __Гранулярность:__ ежедневная

### __Логика сбора комментариев:__ сбор комментариев + ответов к ним.

### __Логика установки временных меток:__ метки привязываются к комментариям через поиск отдельных блоков, которые содержат и текст комментария, и дату. (поиск div блоков с классами комментария и временной метки через XPath селектор) ___________________________________________________________________________________________________________________

## __Описание по функциям__

### ***setup_driver***
#### Создаем и настариваем объект для веб-драйвера с параметрами, которые скрывают признаки автоматизации, помогают обходить защиту от ботов, маскируют парсер под обычного пользователя

### ***close_popups***
#### Я не столкнулась с всплывающими окнами при парсинге, поэтому превентивно добавила список CSS селекторов, которые могут быть зашиты в код сайта.

### ***scroll_pages***
#### Использую плавную прокрутку для имитации поведения человека (1/4 страницы, 1/3, 1/2, до конца). Также проставлена случаная задержка между скроллами. 

### ***parse_dates***
#### Парсим даты на русском из текстового формата в datetime. Прописан словарь по преобразованию месяцев в числовые значения, с помощью регулярных выражений строка приводится в формат дат. 

### ***scrape_messages***
#### Находим все div контейнеры, скролим, парсим текст и дату, дату проверяем на принадлежность к диапазону. Парсинг прекращается, когда доходим до start_date = 01/11/2024.

### ***_main***
#### изначально при переходе между страницами появлялась капча, поэтому я использую f-строки для формирования адреса следующей страницы обсуждения, прерываю работу браузера после парсинга и открываю страницу по новой ссылке. Условие остановки: 1) выход за временной диапазон 2) число страниц больше 150. Результаты сохраняются в датафрейм без дубликатов
_____________________________________________________________________________________________________________________

# Импорты

In [1]:
import time
import random
import re
from datetime import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By

# Проверка для страниц 1 и 2

In [9]:
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    return options

def close_popups(driver):
    popup_selectors = ["button[aria-label*='close']", "button[class*='close']", "div[class*='popup'] button", "div[class*='modal'] button"]
    for selector in popup_selectors:
        close_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
        for button in close_buttons:
            if button.is_displayed():
                driver.execute_script("arguments[0].click();", button)
                time.sleep(random.uniform(0.5, 1.5))

def scroll_page(driver):
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight / 4, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight / 3, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight / 2, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    time.sleep(4)

def parse_date(date_string):
    MONTHS_RU = {
        'янв': 1, 'января': 1, 'фев': 2, 'февр': 2, 'февраля': 2,
        'мар': 3, 'марта': 3, 'апр': 4, 'апреля': 4, 'май': 5, 'мая': 5,
        'июн': 6, 'июня': 6, 'июл': 7, 'июля': 7, 'авг': 8, 'августа': 8,
        'сен': 9, 'сент': 9, 'сентября': 9, 'окт': 10, 'октября': 10,
        'нояб': 11, 'ноября': 11, 'дек': 12, 'декабря': 12
    }
    
    if not date_string:
        return None
    s = date_string.strip().lower()
    m = re.search(r'(\d{1,2})\s+([а-я\.]+)\s+(\d{4})\s*(?:г\.?)?(?:\s*,?\s*(\d{1,2}):(\d{2}))?', s)
    if not m:
        return None
    day = int(m.group(1))
    mon_raw = m.group(2).rstrip('.')
    year = int(m.group(3))
    hour = int(m.group(4)) if m.group(4) else 0
    minute = int(m.group(5)) if m.group(5) else 0
    mon = MONTHS_RU.get(mon_raw) or MONTHS_RU.get(mon_raw[:3])
    if not mon:
        return None
    try:
        return datetime(year, mon, day, hour, minute)
    except ValueError:
        return None

def scrape_message(driver, start_date, end_date):
    containers = driver.find_elements(By.CSS_SELECTOR, "div.border-t")
    page_messages = []
    stop_collecting = False

    for container in containers:
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", container)
        time.sleep(0.1)
        comment_cont = container.find_elements(By.XPATH, ".//div[.//div[contains(@class,'break-words') and contains(@class,'leading-5')]"
            " and .//span[@data-test='comment-date']]")

        for cont in comment_cont:
            try:
                msg_el = cont.find_element(By.CSS_SELECTOR, "div.break-words.leading-5")
                time_el = cont.find_element(By.CSS_SELECTOR, "span[data-test='comment-date']")
            except Exception:
                continue

            msg_text = msg_el.text.strip()
            time_text = time_el.text.strip()

            message_date = parse_date(time_text)
            
            if message_date and message_date < start_date:
                stop_collecting = True
                break

            if message_date and start_date <= message_date <= end_date:
                page_messages.append({'дата': message_date,'сообщение': msg_text})

        if stop_collecting:
            break

    return page_messages, stop_collecting

def main():
    all_messages = []
   
    start_date = datetime(2024, 11, 1)  
    end_date = datetime(2025, 11, 1)   
    current_page = 1
    max_pages = 2
    stop_collecting = False
    
    options = setup_driver()

    while current_page <= max_pages and not stop_collecting:
        driver = webdriver.Chrome(options=options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        driver.maximize_window()

        if current_page == 1:
            url = 'https://ru.investing.com/crypto/ethereum/chat'
        else:
            url = f'https://ru.investing.com/crypto/ethereum/chat/{current_page}'

        driver.get(url)
        time.sleep(8)

        close_popups(driver)
        scroll_page(driver)
        page_messages, stop_collecting = scrape_message(driver, start_date, end_date)
        
        all_messages.extend(page_messages)
        driver.quit()

        if not stop_collecting:
            current_page += 1
            time.sleep(random.uniform(2, 5))

    df_final = pd.DataFrame(all_messages).drop_duplicates()
    return df_final
    
df_final = main()
display(df_final)

,дата,сообщение
0,2025-10-31 06:38:00,пойдет до 4200?
1,2025-10-31 05:07:00,Быстрый возврат к 4000 . Паничка у шортиков )
2,2025-10-31 17:41:00,Быстрый возврат к 3500 ты имел ввиду?
4,2025-10-30 22:45:00,На 3300-3500?
5,2025-10-30 13:54:00,"Зачем валидаторы продают сейчас, если можно пр..."
6,2025-10-30 15:46:00,потому что не хотят продавать за 1800
8,2025-10-29 23:03:00,"Куда дальше? Кто знает, предполагает?"
9,2025-10-30 05:09:00,"Вниз,тут же по графику видно,если зояешл лонго..."
11,2025-10-28 03:48:00,16-1(2)6
12,2025-10-27 11:18:00,$ETHUSD : new target 4994


# Основной парсер комментарий за год (парсинг до даты)

In [3]:
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    return options

def close_popups(driver):
    popup_selectors = ["button[aria-label*='close']", "button[class*='close']", "div[class*='popup'] button", "div[class*='modal'] button"]#".popupCloseIcon",".largeBannerCloser"]
    for selector in popup_selectors:
        close_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
        for button in close_buttons:
            if button.is_displayed():
                driver.execute_script("arguments[0].click();", button)
                time.sleep(random.uniform(0.5, 1.5))

def scroll_page(driver):
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight / 4, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight / 3, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight / 2, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight, behavior: 'smooth'});")
    time.sleep(random.uniform(0.5, 1.0))
    time.sleep(4)

def parse_date(date_string):
    MONTHS_RU = {
        'янв': 1, 'января': 1, 'фев': 2, 'февр': 2, 'февраля': 2,
        'мар': 3, 'марта': 3, 'апр': 4, 'апреля': 4, 'май': 5, 'мая': 5,
        'июн': 6, 'июня': 6, 'июл': 7, 'июля': 7, 'авг': 8, 'августа': 8,
        'сен': 9, 'сент': 9, 'сентября': 9, 'окт': 10, 'октября': 10,
        'нояб': 11, 'ноября': 11, 'дек': 12, 'декабря': 12
    }
    
    if not date_string:
        return None
    s = date_string.strip().lower()
    m = re.search(r'(\d{1,2})\s+([а-я\.]+)\s+(\d{4})\s*(?:г\.?)?(?:\s*,?\s*(\d{1,2}):(\d{2}))?', s)
    if not m:
        return None
    day = int(m.group(1))
    mon_raw = m.group(2).rstrip('.')
    year = int(m.group(3))
    hour = int(m.group(4)) if m.group(4) else 0
    minute = int(m.group(5)) if m.group(5) else 0
    mon = MONTHS_RU.get(mon_raw) or MONTHS_RU.get(mon_raw[:3])
    if not mon:
        return None
    try:
        return datetime(year, mon, day, hour, minute)
    except ValueError:
        return None

def scrape_messages(driver, start_date, end_date):
    containers = driver.find_elements(By.CSS_SELECTOR, "div.border-t")
    page_messages = []
    stop_collecting = False

    for container in containers:
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", container)
        time.sleep(0.1)
        comment_cont = container.find_elements(By.XPATH, ".//div[.//div[contains(@class,'break-words') and contains(@class,'leading-5')]"
            " and .//span[@data-test='comment-date']]")

        for cont in comment_cont:
            try:
                msg_el = cont.find_element(By.CSS_SELECTOR, "div.break-words.leading-5")
                time_el = cont.find_element(By.CSS_SELECTOR, "span[data-test='comment-date']")
            except Exception:
                continue

            msg_text = msg_el.text.strip()
            time_text = time_el.text.strip()

            message_date = parse_date(time_text)
            

            if message_date and message_date < start_date:
                stop_collecting = True
                break

            if message_date and start_date <= message_date <= end_date:
                page_messages.append({'дата': message_date,'сообщение': msg_text})

        if stop_collecting:
            break

    return page_messages, stop_collecting

def main():
    all_messages = []
   
    start_date = datetime(2024, 11, 1)  
    end_date = datetime(2025, 11, 1)   
    current_page = 1
    max_pages = 150
    stop_collecting = False
    
    options = setup_driver()

    while current_page <= max_pages and not stop_collecting:
        driver = webdriver.Chrome(options=options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        driver.maximize_window()

        if current_page == 1:
            url = 'https://ru.investing.com/crypto/ethereum/chat'
        else:
            url = f'https://ru.investing.com/crypto/ethereum/chat/{current_page}'

        driver.get(url)
        time.sleep(8)

        close_popups(driver)
        scroll_page(driver)
        page_messages, stop_collecting = scrape_messages(driver, start_date, end_date)
        
        all_messages.extend(page_messages)
        driver.quit()

        if not stop_collecting:
            current_page += 1
            time.sleep(random.uniform(2, 5))

    df_final = pd.DataFrame(all_messages).drop_duplicates()
    return df_final

In [5]:
df_final = main()

In [ ]:
df_final.to_excel('ethereum_data.xlsx',index=False)